# Releitura independente das evidências
Este notebook só consulta registros existentes. Não cria schemas, tabelas ou execuções.

Cole em `run_ids` a lista JSON emitida pelo notebook 02. Execute em uma nova sessão Python para comprovar a independência do estado anterior. Se reutilizar a sessão atual, a verificação continua útil, mas não comprova uma nova sessão.

Resultado esperado: SUCESSO, SUCESSO, FALHOU (falha controlada), com hashes válidos.

**Parâmetros:** execute apenas a primeira célula de código para exibir os widgets. Preencha os campos e depois execute as demais células. Use **Run all** quando os parâmetros já estiverem preenchidos.


In [0]:
# catalogo: workspace, o mesmo catálogo usado na nova execução do notebook 02.
# run_ids: obrigatório. Cole a lista JSON completa de três IDs emitida pelo novo notebook 02.
# Formato: ["UUID_1", "UUID_2", "UUID_3"], mantendo a ordem. Não use estes placeholders.
# Não reutilize os IDs da validação antiga nesta rodada; queremos validar as novas evidências.
# Execute em uma nova sessão Python para verificar a persistência independentemente da sessão anterior.
# Execute somente este CMD, preencha os widgets e então prossiga.
# 1. Exibir parâmetros — execute esta célula antes de preencher os campos.
dbutils.widgets.text("catalogo", spark.catalog.currentCatalog())
dbutils.widgets.text("run_ids", "[]")

print("Preencha os widgets acima e depois execute as próximas células.")


In [0]:
# CMD 2 — Localizar a pasta Git e importar o pacote na nova sessão. Nenhum preenchimento adicional.
from pathlib import Path
import sys
atual = Path.cwd().resolve()
raiz = next((p for p in [atual, *atual.parents]
             if (p / "src" / "medallion").is_dir() and (p / "pyproject.toml").is_file()), None)
if raiz is None:
    raise RuntimeError("Abra este notebook na pasta Git; se necessário, use %cd com a raiz do projeto.")
if str(raiz / "src") not in sys.path:
    sys.path.insert(0, str(raiz / "src"))
from medallion.config import PipelineConfig



In [0]:
# CMD 3 — Ler run_ids e verificar evidências existentes. Esperado: RELEITURA DAS EVIDÊNCIAS: OK.
import json
from medallion.observability.audit_delta import DeltaAuditSink

ids = json.loads(dbutils.widgets.get("run_ids"))
if not isinstance(ids, list) or len(ids) != 3 or any(not isinstance(x, str) for x in ids):
    raise ValueError("Cole a lista JSON com os três run_ids do notebook 02.")
if len(set(ids)) != 3:
    raise ValueError("Informe três tentativas distintas.")
config = PipelineConfig(catalogo=dbutils.widgets.get("catalogo").strip())
sink = DeltaAuditSink(spark, config)
resultado = [sink.verificar(run_id) for run_id in ids]
assert [r["status"] for r in resultado] == ["SUCESSO", "SUCESSO", "FALHOU"]
for registro in resultado:
    print(registro)
print("RELEITURA DAS EVIDÊNCIAS: OK")



## Coleta desta revalidação

Após executar todos os CMDs, confira a mensagem final esperada. Se ocorrer erro, preserve o output e interrompa a sequência.

Exporte este notebook como **IPYNB com outputs**, abra o arquivo e confirme que os resultados estão presentes. Salve o arquivo antes de limpar outputs ou mudar a sessão. Faça commit/push na branch `chore/estrutura-inicial`, registre o SHA e envie o export para revisão antes de avançar ao próximo notebook.

Sugestão de commit: `docs: revalida 03_reler_evidencias com instrucoes de parametros`.

O commit sozinho não comprova que os outputs foram incluídos. A cópia pública do export será revisada para remover e-mails e caminhos pessoais, com manifesto e hashes, preservando a coleta anterior. Esta rodada só será registrada como validada após revisão do novo export.
